# Quetion 7




# Databricks Asset Bundle (DAB) CI/CD Deployment Guide
```
## 1. Executive Summary
This document details the automated CI/CD pipeline setup for Databricks Asset Bundles (DAB) using GitHub Actions. The architecture separates code validation from production deployment using a two-tier branch workflow (`validate-bundle-branch` → `main-2`).

---

## 2. CI/CD Pipeline Architecture

The pipeline consists of two distinct workflows stored in `.github/workflows/`:

* **`validate.yml`**: Triggers automatically when a Pull Request is opened from `validate-bundle-branch` targeting `main-2`. It executes standard syntax and configuration checks via `databricks bundle validate`.
* **`deploy.yml`**: Triggers automatically upon merging the Pull Request into `main-2`. It executes the production bundle deployment via `databricks bundle deploy --target prod`.


```

[Feature Branch: validate-bundle-branch]
│
│ (Create Pull Request targeting main-2)
▼
┌───────────────────┐
│   validate.yml    │  ──► Runs `databricks bundle validate`
└───────────────────┘
│
│ (PR Merged into main-2)
▼
┌───────────────────┐
│    deploy.yml     │  ──► Runs `databricks bundle deploy --target prod`
└───────────────────┘

```

---

## 3. Workflow Configurations

### A. Validation Workflow (`.github/workflows/validate.yml`)
```yaml
name: Validate Databricks Bundle

on:
  pull_request:
    branches:
      - main-2

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Code
        uses: actions/checkout@v3

      - name: Setup Databricks CLI
        uses: databricks/setup-cli@v0.2.1

      - name: Validate Bundle Syntax
        run: databricks bundle validate

```

### B. Production Deployment Workflow (`.github/workflows/deploy.yml`)

```yaml
name: Deploy Production Bundle

on:
  push:
    branches:
      - main-2  # Executed on PR merge into main-2

jobs:
  deploy:
    runs-on: ubuntu-latest
    # Explicit permissions required for OIDC token emission & repository read
    permissions:
      id-token: write
      contents: read

    steps:
      - name: Checkout Code
        uses: actions/checkout@v3

      - name: Setup Databricks CLI
        uses: databricks/setup-cli@v0.2.1

      - name: Deploy Bundle to Prod
        env:
          DATABRICKS_HOST: [https://dbc-2bd5e4bd-405a.cloud.databricks.com](https://dbc-2bd5e4bd-405a.cloud.databricks.com)
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        run: |
          databricks bundle deploy --target prod

```

---

## 4. Databricks Bundle Configuration (`databricks.yml`)

The target deployment path is configured outside personal workspace user directories to prevent workspace lock issues and guarantee persistent execution across team members.

```yaml
bundle:
  name: my_project

targets:
  prod:
    mode: production
    workspace:
      host: [https://dbc-2bd5e4bd-405a.cloud.databricks.com](https://dbc-2bd5e4bd-405a.cloud.databricks.com)
      root_path: /Shared/prod_deployments/${bundle.name}/${bundle.target}
    variables:
      catalog: prod
      schema: bronze
    permissions:
      - service_principal_name: 0c28e675-641f-49ef-81a9-b5625f2257b4
        level: CAN_MANAGE

```

---

## 5. Authentication & Technical Constraints

### A. OIDC Token Permissions (`id-token: write`)

The `deploy.yml` workflow includes `permissions: id-token: write` to allow the GitHub Actions runner to request short-lived OpenID Connect (OIDC) JSON Web Tokens (JWTs) for Federated Identity checks.

### B. Why "Pure / Keyless OIDC" (Zero Secrets) Was Not Feasible

1. **Account-Level Workload Identity Federation:** Fully keyless OIDC (without client secrets) requires configuring a federated trust relationship between GitHub and the Cloud Provider (Azure AD / AWS IAM / GCP Workload Identity) at the **Databricks Account Console level**. Standard Workspace-level environments lack direct keyless token exchange endpoints.
2. **Databricks CLI OAuth M2M Requirement:** Databricks Asset Bundles rely on the Databricks CLI, which uses OAuth 2.0 Machine-to-Machine (M2M) authentication. The CLI requires explicit OAuth credentials (`DATABRICKS_CLIENT_ID` and `DATABRICKS_CLIENT_SECRET`) to maintain deployment sessions.
3. **Implemented Solution:** Rather than using Personal Access Tokens (PATs), authentication was configured using a dedicated non-human **Service Principal**. This decouples the deployment pipeline from individual user accounts, ensuring non-blocking, production-grade automated execution.

```

```

# Question 8

Design a rollback plan: if a bundle deploy to prod breaks a job, what CLI commands would you run to
redeploy the previous working version quickly?


# Production Rollback Plan – Databricks Asset Bundles

## Objective

If a new Databricks Bundle deployment breaks a production job, the previous working version can be quickly restored by identifying the last known-good Git commit and redeploying that version to the production workspace.

## Rollback Steps

### 1. Identify the Last Working Commit

Check the Git commit history:

```bash
git log --oneline
```

Example:

```text
a8f32cd Fix production job
72bc91a Add sales transformation
45de123 Initial bundle
```

If `a8f32cd` caused the issue and `72bc91a` was the last working version, use `72bc91a` for the rollback.

### 2. Checkout the Previous Working Version

```bash
git checkout 72bc91a
```

This changes the local project to the state of the previous working commit.

### 3. Validate the Bundle

Before deploying the rollback version, validate the bundle against the production target:

```bash
databricks bundle validate --target prod
```

If validation is successful, proceed with the deployment.

### 4. Redeploy the Previous Working Version

Deploy the previous version to the production workspace:

```bash
databricks bundle deploy --target prod
```

This restores the Databricks resources defined in the previous working bundle version.

### 5. Verify the Production Job

After deployment, verify that the production job is working correctly.

Jobs can be listed using:

```bash
databricks jobs list
```

If required, trigger the job manually:

```bash
databricks jobs run-now <JOB_ID>
```

## Rollback Flow

```text
Production Deployment
        ↓
Job Breaks
        ↓
Check Git History
        ↓
Identify Last Working Commit
        ↓
git checkout <working-commit>
        ↓
Bundle Validation
        ↓
databricks bundle validate --target prod
        ↓
Redeploy to Production
        ↓
databricks bundle deploy --target prod
        ↓
Verify / Run Job
```

## Important Note

Checking out an old Git commit only changes the local project version. It does **not** rollback the production workspace by itself.

The actual rollback happens when the previous working version is redeployed using:

```bash
databricks bundle deploy --target prod
```

## Recommended Practice

The rollback should be based on a known-good Git commit rather than manually modifying production resources. This keeps the production environment consistent with version-controlled code and provides a clear audit trail of the deployed version.



# Question 9
Write a one-page onboarding guide for a new team member explaining how a change moves from a
local databricks.yml edit to running safely in production, referencing the CLI, the bundle lifecycle, and
the CI/CD workflow together.


# Developer Guide: Databricks Asset Bundle (DAB) Deployment Lifecycle

This guide details the complete path a code or infrastructure update takes—from your local environment to safe execution in production—using Databricks Asset Bundles (DABs), the Databricks CLI, and integrated CI/CD pipelines.

---

## 1. Local Development & Configuration

Every change begins in your local repository where `databricks.yml` defines the bundle configuration (tasks, cluster specs, job settings, and target environments).

* **Bundle Validation:** Verify syntax and environment references before pushing:
```bash
databricks bundle validate

```


* **Development Deployment:** Test your logic isolated in your personal sandbox:
```bash
databricks bundle deploy --target dev

```


*This package builds and deploys artifacts to your individual development workspace without affecting shared resources.*
* **Dev Execution:** Run your workflow on demand to test execution:
```bash
databricks bundle run <job-key> --target dev

```



---

## 2. Source Control & Automated Validation (Pull Request)

Once local execution succeeds, commit your code and open a Pull Request (PR) against the `main` or `release` branch. Opening a PR triggers the automated continuous integration workflow.

* **Linting & Testing:** Unit tests run alongside `databricks bundle validate` to ensure structural validity.
* **Dry Run Verification:** The CI pipeline runs a dry-run deployment targeting the staging environment:
```bash
databricks bundle deploy --target staging --dry-run

```


* **Peer Review:** Team members review the code and configuration diffs. Approval is required before merging.

---

## 3. Staging Deployment & Automated Testing

Upon merging the PR into the integration branch, the Continuous Integration / Continuous Deployment (CI/CD) pipeline takes over deployment to the Staging environment.

1. **Deploy to Staging:**
```bash
databricks bundle deploy --target staging

```


2. **Integration Testing:** Automated integration and regression tests run against non-production data or isolated catalogs in Unity Catalog.
3. **Artifact Immutability:** The deployment package compiled during this stage acts as the candidate release.

---

## 4. Production Release & Safe Execution

Deploying to production requires strict governance and automated quality gates.

```
+------------------+      +-------------------+      +------------------+      +-------------------+
|  Local Developer | ---> |   Pull Request    | ---> |     Staging      | ---> |    Production     |
| (bundle validate |      | (CI Validation /  |      |  (Auto-Deploy &  |      |   (Gated CD &     |
|   & dev deploy)  |      |   Dry-Run Check)  |      |   Integration)   |      | Automated Schedule|
+------------------+      +-------------------+      +------------------+      +-------------------+

```

1. **Production Deployment Trigger:** CD pipelines execute deployment upon a release tag or manual approval gate:
```bash
databricks bundle deploy --target prod

```


2. **Unity Catalog & Permissions:** Assets run under a dedicated Service Principal identity rather than individual developer accounts to ensure strict access control and auditability.
3. **Production Run & Monitoring:** Scheduled jobs and Lakeflow pipelines execute automatically under production cluster configs. Telemetry, alerting, and failure notifications monitor run state continuously.